<a href="https://colab.research.google.com/github/kzdanowski/KGN_Programowanie2/blob/main/P2Lab02_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Ćwiczenia z Programowanie 2, język SQL, operacje na tabelach.**

Podstawwowe operacje na tabelach:
- operacje teoriomnogościowe: suma, iloczyn, różnica
- rzutowania na część kolumn,
- złączenia,
- funkcje grupujące.

Struktura kwerendy SQL:   
SELECT *nazwy i definicje kolumn*   
FROM *nazwy tabel i sposoby ich złączenia*  
WHERE *warunek selekcji wierszy*  
GROUP BY *kolumny po których grupujemy wyniki*  
HAVING *filtrowanie wyników po grupowaniu*  
ORDER BY *kolumny, po których sortujemy* [ASC|DESC];


## Kolejność wykonywania
- FROM - tworzy złączenia tabel w jedną tabelę,
- WHERE - odrzuca wiersze nie spełniające warunku,
- GROUP BY *atrybuty* - grupuje wynik po tej samej wartosci *atrybuty*,
- HAVING - filtruje wiersze na podstawie wartości funkcji grupujących.

## Złączenia tabel
- NATURAL JOIN - złączenie po wspólnej kolumnie,
do złączenia brane są krotki, które mają tę samą wartość we wspólnych kolumnach,
- JOIN - *T1 JOIN T2 ON T1.Atr1 = T2.Atr2*, (także określany INNER JOIN),
- FULL OUTER JOIN   - niepasujące wiersze zostaną uzupełnione wartościami null,
- LEFT OUTER JOIN - niepasujące wiersze z lewej tabeli zostaną uzupełnione wartościami null,
- RIGHT OUTER JOIN - analogicznie
- CROSS JOIN - iloczyn kartezjański.

SELECT * FROM A CROSS JOIN B;  
jest równoważne  
SELECT * FROM A, B;


## Funkcje grupujące
- count(*) lub count(atrybut) - zliczanie,  
  a) count(*) zlicza wszystkie wiersze dla danej wartości grupy,      
  b) count(atrybut) zlicza wiersze, gdzie atrybut jest różny od null,
- avg(atrybut) - średnia,
- max(atrybut), min(atrybut).

In [ ]:
import pandas as pd
import sqlite3

# Tabela 1: Dane osobowe
df_studenci = pd.DataFrame({
    'id_stud': [101, 102, 103, 104],
    'imie': ['Adam', 'Beata', 'Cezary', 'Daria'],
    'kierunek': ['Filozofia', 'Kognitywistyka', 'Filozofia', 'Informatyka']
})

# Tabela 2: Wyniki egzaminów
df_oceny = pd.DataFrame({
    'id_stud': [101, 101, 102, 103, 103, 105], # 105 to student spoza listy
    'przedmiot': ['Logika I', 'Etyka', 'Logika I', 'Logika I', 'Metafizyka', 'Logika I'],
    'ocena': [5.0, 4.0, 4.5, 3.0, 3.5, 2.0]
})

# Tabela 3: Wykladowcy

df_wykladowcy = pd.DataFrame({
    'id_wykl': [101, 102, 103], # 105 to student spoza listy
    'nazwisko': ['Abacki', 'Babacki', 'Cabacki']
})

# Tabela 4: Grafik wykladow

df_grafik = pd.DataFrame({
    'id_wykl': [101, 102, 103, 103 ],
    'przedmiot': ['Logika I', 'Etyka', 'Metafizyka', 'Logika I']
})

conn = sqlite3.connect('uczelnia.db')
df_studenci.to_sql('studenci', conn, index=False, if_exists='replace')
df_oceny.to_sql('oceny', conn, index=False, if_exists='replace')
df_wykladowcy.to_sql('wykladowcy', conn, index=False, if_exists='replace')
df_grafik.to_sql('grafik', conn, index=False, if_exists='replace')
conn.close()


In [ ]:
!pip install jupysql --quiet
%load_ext sql
%config SqlMagic.autopandas = True  # Wyniki będą od razu ramkami Pandas
%sql sqlite:///prosta_baza.db

%sql sqlite:///uczelnia.db

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 4.2 MB/s eta 0:00:00


Connecting to 'sqlite:///prosta_baza.db'

Connecting and switching to connection 'sqlite:///uczelnia.db'

In [ ]:
%%sql
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;


Running query in 'sqlite:///uczelnia.db'

,name
0,grafik
1,oceny
2,studenci
3,wykladowcy


In [ ]:
%%sql
PRAGMA table_info(studenci);

Running query in 'sqlite:///uczelnia.db'

,cid,name,type,notnull,dflt_value,pk
0,0,id_stud,INTEGER,0,None,0
1,1,imie,TEXT,0,None,0
2,2,kierunek,TEXT,0,None,0


In [ ]:
%%sql

SELECT
    s.imie,
    s.kierunek,
    COUNT(o.ocena) AS liczba_egzaminow,
    ROUND(AVG(o.ocena), 2) AS srednia
FROM studenci s
LEFT JOIN oceny o ON s.id_stud = o.id_stud
GROUP BY s.id_stud;

Running query in 'sqlite:///uczelnia.db'

,imie,kierunek,liczba_egzaminow,srednia
0,Adam,Filozofia,2,4.50
1,Beata,Kognitywistyka,1,4.50
2,Cezary,Filozofia,2,3.25
3,Daria,Informatyka,0,NaN


In [ ]:
%%sql

# Tutaj INNER JOIN działa jak iloczyn kartezjański (CROSS JOIN)
# bo brakuje warunku definiującego złączenie.

SELECT *
FROM wykladowcy INNER JOIN grafik;

Running query in 'sqlite:///uczelnia.db'

,id_wykl,nazwisko,id_wykl,przedmiot
0,101,Abacki,101,Logika I
1,101,Abacki,102,Etyka
2,101,Abacki,103,Metafizyka
3,101,Abacki,103,Logika I
4,102,Babacki,101,Logika I
5,102,Babacki,102,Etyka
6,102,Babacki,103,Metafizyka
7,102,Babacki,103,Logika I
8,103,Cabacki,101,Logika I
9,103,Cabacki,102,Etyka


In [ ]:
%%sql
SELECT *
FROM Wykladowcy, Grafik;

Running query in 'sqlite:///uczelnia.db'

,id_wykl,nazwisko,id_wykl,przedmiot
0,101,Abacki,101,Logika I
1,101,Abacki,102,Etyka
2,101,Abacki,103,Metafizyka
3,101,Abacki,103,Logika I
4,102,Babacki,101,Logika I
5,102,Babacki,102,Etyka
6,102,Babacki,103,Metafizyka
7,102,Babacki,103,Logika I
8,103,Cabacki,101,Logika I
9,103,Cabacki,102,Etyka


### Napisz następujące kwerendy:
- policz średnią wszystkich ocen,
- policz średnią ocen każdego studenta oddzielnie,
- policz ile kursów może prowadzić każdy wykładowca,
- połącz studentów i wykładowców, u których student mógł brać kurs,
- policz średnią ocen z każdego przedmiotu,
- policz ile kursów brał każdy student,
- wypisz studentów, których średnia ocen jest większa niż 3,
- wypisz przedmioty, na które uczęszczali studenci przynajmniej dwóch kierunków.

In [ ]:
%%sql
SELECT imie as "Imię studenta"
FROM studenci;

Running query in 'sqlite:///uczelnia.db'

,Imię studenta
0,Adam
1,Beata
2,Cezary
3,Daria
